# Tiny Text-to-Pixel Generative AI
### A from-scratch Conditional VAE for a 1-hour class

Goal:

**text prompt + random latent vector → new 16×16 pixel art**

The dataset contains compositional prompts such as `red small robot`. Five combinations are deliberately hidden from training so we can test whether the model can combine concepts it learned separately.


## 1. Build the dataset
The dataset generator is downloaded from this repository and creates the same 3,050 teaching images automatically.


In [ ]:
# Build the teaching dataset reproducibly from the repository
!wget -q -O generate_dataset.py https://raw.githubusercontent.com/dujing82-blip/tiny-text-to-pixel-ai/main/generate_dataset.py
!rm -rf tiny_pixel_dataset
!python generate_dataset.py


## 2. Load images and captions


In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image

BASE = "tiny_pixel_dataset"
df = pd.read_csv(os.path.join(BASE, "captions.csv"))
train_df = df[df["split"]=="train"].reset_index(drop=True)
test_df  = df[df["split"]=="unseen_test"].reset_index(drop=True)

def load_images(frame):
    out=[]
    for fn in frame["filename"]:
        im = Image.open(os.path.join(BASE, fn)).convert("RGB")
        out.append(np.asarray(im, dtype=np.float32)/255.0)
    return np.stack(out)

X_train = load_images(train_df)
X_test  = load_images(test_df)

print("Training:", X_train.shape)
print("Held-out:", X_test.shape)
display(train_df.head())


## 3. Words become numbers
We use **three separate concepts**: color, size, and object. Each gets its own embedding.

This is deliberately simple: the model is not a language model. It is a tiny demonstration of conditioning.


In [ ]:
colors = ["red","blue","green","yellow"]
sizes = ["small","big"]
objects = ["heart","robot","spaceship","tree","house"]

color_to_id = {w:i for i,w in enumerate(colors)}
size_to_id = {w:i for i,w in enumerate(sizes)}
object_to_id = {w:i for i,w in enumerate(objects)}

def ids(frame):
    return (frame["color"].map(color_to_id).to_numpy(np.int32), frame["size"].map(size_to_id).to_numpy(np.int32), frame["object"].map(object_to_id).to_numpy(np.int32))

c_train, s_train, o_train = ids(train_df)


## 4. Encoder: image + text concepts → latent distribution


In [ ]:
latent_dim = 8
embed_dim = 4
image_in = keras.Input((16,16,3), name="image")
color_in = keras.Input((), dtype="int32", name="color")
size_in  = keras.Input((), dtype="int32", name="size")
object_in= keras.Input((), dtype="int32", name="object")
img = layers.Flatten()(image_in)
ce = layers.Flatten()(layers.Embedding(len(colors), embed_dim)(color_in))
se = layers.Flatten()(layers.Embedding(len(sizes), embed_dim)(size_in))
oe = layers.Flatten()(layers.Embedding(len(objects), embed_dim)(object_in))
x = layers.Concatenate()([img, ce, se, oe])
x = layers.Dense(128, activation="relu")(x)
x = layers.Dense(64, activation="relu")(x)
z_mean = layers.Dense(latent_dim)(x)
z_log_var = layers.Dense(latent_dim)(x)
class Sampling(layers.Layer):
    def call(self, inputs):
        mean, log_var = inputs
        eps = tf.random.normal(tf.shape(mean))
        return mean + tf.exp(0.5 * log_var) * eps
z = Sampling()([z_mean, z_log_var])
encoder = keras.Model([image_in,color_in,size_in,object_in],[z_mean,z_log_var,z], name="encoder")
encoder.summary()


## 5. Decoder / Generator: random latent + text concepts → pixels


In [ ]:
z_in = keras.Input((latent_dim,), name="z")
dc = keras.Input((), dtype="int32", name="color")
ds = keras.Input((), dtype="int32", name="size")
do = keras.Input((), dtype="int32", name="object")
ce = layers.Flatten()(layers.Embedding(len(colors), embed_dim)(dc))
se = layers.Flatten()(layers.Embedding(len(sizes), embed_dim)(ds))
oe = layers.Flatten()(layers.Embedding(len(objects), embed_dim)(do))
x = layers.Concatenate()([z_in, ce, se, oe])
x = layers.Dense(64, activation="relu")(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dense(16*16*3, activation="sigmoid")(x)
out = layers.Reshape((16,16,3))(x)
decoder = keras.Model([z_in,dc,ds,do], out, name="decoder")
decoder.summary()


## 6. Conditional VAE training


In [ ]:
class CVAE(keras.Model):
    def __init__(self, encoder, decoder, beta=0.02):
        super().__init__(); self.encoder=encoder; self.decoder=decoder; self.beta=beta
    def train_step(self, data):
        images, c, s, o = data
        with tf.GradientTape() as tape:
            mean, log_var, z = self.encoder([images,c,s,o], training=True)
            recon = self.decoder([z,c,s,o], training=True)
            rec = tf.reduce_mean(tf.reduce_sum(tf.square(images-recon), axis=[1,2,3]))
            kl = -0.5 * tf.reduce_mean(tf.reduce_sum(1 + log_var - tf.square(mean) - tf.exp(log_var), axis=1))
            loss = rec + self.beta*kl
        grads=tape.gradient(loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads,self.trainable_weights))
        return {"loss":loss, "reconstruction":rec, "KL":kl}
train_ds = tf.data.Dataset.from_tensor_slices((X_train,c_train,s_train,o_train)).shuffle(len(X_train)).batch(64)
vae=CVAE(encoder,decoder)
vae.compile(optimizer=keras.optimizers.Adam(1e-3))
history=vae.fit(train_ds, epochs=60)


## 7. Type a prompt and generate
Try the same prompt repeatedly. The words stay fixed; the random latent vector changes.


In [ ]:
def parse_prompt(prompt):
    words = prompt.lower().split()
    color = next((w for w in words if w in color_to_id), None)
    size  = next((w for w in words if w in size_to_id), None)
    obj   = next((w for w in words if w in object_to_id), None)
    if None in (color,size,obj): raise ValueError("Use one color + one size + one object.")
    return color,size,obj
def generate(prompt, n=8):
    color,size,obj=parse_prompt(prompt)
    z=np.random.normal(size=(n,latent_dim)).astype("float32")
    c=np.full(n,color_to_id[color],np.int32); s=np.full(n,size_to_id[size],np.int32); o=np.full(n,object_to_id[obj],np.int32)
    imgs=decoder.predict([z,c,s,o],verbose=0)
    plt.figure(figsize=(12,2))
    for i,img in enumerate(imgs):
        plt.subplot(1,n,i+1); plt.imshow(np.clip(img,0,1), interpolation="nearest"); plt.axis("off")
    plt.suptitle(prompt); plt.show()
generate("red big robot")


## 8. The key experiment: generate a combination AI never saw
These five combinations were excluded from training:

- blue big spaceship
- yellow small robot
- green big heart
- red small tree
- blue small house

The model has seen each **word/concept** separately, but never these exact combinations.


In [ ]:
generate("blue big spaceship")
generate("yellow small robot")
generate("green big heart")


## 9. Interactive mini text-to-image app
This is intentionally a tiny controlled vocabulary, not full natural-language understanding.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
box=widgets.Text(value="blue big spaceship", description="Prompt:")
button=widgets.Button(description="Generate", button_style="success")
out=widgets.Output()
def clicked(_):
    with out:
        clear_output(wait=True)
        try: generate(box.value, n=8)
        except Exception as e: print(e); print("Example: red small robot")
button.on_click(clicked)
display(box,button,out)


# What did we build?

**words → embeddings**

**random latent vector + embeddings → pixels**

Modern text-to-image systems are vastly larger and use sophisticated text encoders and diffusion/flow models, but conditioning a generative image model on a representation of text is one of the central ideas students should leave with.
